In [ ]:
"""
data_preprocessing.py

Prepares Sentinel-2, ECOSTRESS, NASADEM, target LST,
and thermal conductivity data for the U-Net PINN.

U-Net input channels:

    1. Sentinel-2 B2
    2. Sentinel-2 B3
    3. Sentinel-2 B4
    4. Sentinel-2 B8
    5. Sentinel-2 B11
    6. Sentinel-2 B12
    7. NDVI
    8. NDBI
    9. NASADEM elevation
    10. ECOSTRESS LST

Final arrays:

    X = (N, 10, 256, 256)
    Y = (N, 1, 256, 256)
    K = (N, 1, 256, 256)

X:
    Inputs to the U-Net.

Y:
    Reference LST used for data loss.

K:
    Thermal conductivity used by HeatDiffusionPINN.
"""

from pathlib import Path

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject


# ============================================================
# Paths
# ============================================================

DATA_DIR = Path(
    "/content/drive/MyDrive/SISTER/data"
)

SENTINEL_DIR = DATA_DIR / "Sentinel-2-2A"
ECOSTRESS_DIR = DATA_DIR / "ECOSTRESS"
ELEVATION_DIR = DATA_DIR / "Elevation"

# Create these folders/files when your team has finalized
# the corresponding datasets.
TARGET_DIR = DATA_DIR / "Target-LST"
CONDUCTIVITY_DIR = DATA_DIR / "Conductivity"

PROCESSED_DIR = DATA_DIR / "processed"


# ============================================================
# Configuration
# ============================================================

PATCH_SIZE = 256
STRIDE = 256

# Sentinel-2 export is on a 10 m grid.
# This must match the spatial grid used by the U-Net output.
PIXEL_SIZE = 10.0


# ============================================================
# Find GeoTIFF files
# ============================================================

def find_tif_files(folder):
    """
    Return all GeoTIFF files in a folder.
    """

    if not folder.exists():
        return []

    return sorted(
        list(folder.glob("*.tif"))
        + list(folder.glob("*.tiff"))
    )


def show_available_files():
    """
    Print all available GeoTIFF files.
    """

    folders = {
        "Sentinel-2": SENTINEL_DIR,
        "ECOSTRESS": ECOSTRESS_DIR,
        "Elevation": ELEVATION_DIR,
        "Target LST": TARGET_DIR,
        "Conductivity": CONDUCTIVITY_DIR
    }

    for name, folder in folders.items():

        print(f"\n{name}")
        print("=" * len(name))

        files = find_tif_files(folder)

        if not files:

            print(
                f"No GeoTIFF files found in:\n{folder}"
            )

        else:

            for index, file in enumerate(files):

                print(
                    f"[{index}] {file.name}"
                )


# ============================================================
# Select a file
# ============================================================

def select_file(
    folder,
    name
):
    """
    Select the first GeoTIFF in a folder.

    This is convenient for testing.

    For the final experiment, exact files should be supplied
    so that Sentinel-2, ECOSTRESS, target LST, and conductivity
    correspond to the same scene/date.
    """

    files = find_tif_files(
        folder
    )

    if not files:

        raise FileNotFoundError(
            f"No {name} GeoTIFF was found in:\n"
            f"{folder}"
        )

    if len(files) > 1:

        print(
            f"\nWARNING: {len(files)} {name} files found."
        )

        print(
            f"Using: {files[0].name}"
        )

        print(
            "For final training, explicitly match "
            "the same date/scene across datasets."
        )

    return files[0]


# ============================================================
# Read GeoTIFF
# ============================================================

def read_raster(path):
    """
    Read a GeoTIFF.

    Returns:

        data:
            Shape (channels, height, width)

        profile:
            Raster metadata.
    """

    with rasterio.open(path) as src:

        data = src.read().astype(
            np.float32
        )

        profile = src.profile.copy()

    return data, profile


# ============================================================
# Align raster to Sentinel-2 grid
# ============================================================

def align_to_reference(
    source_path,
    reference_path,
    resampling_method
):
    """
    Reproject and resample a source raster so that it matches
    the Sentinel-2 grid exactly.

    The reference determines:

        - height
        - width
        - CRS
        - spatial transform
    """

    with rasterio.open(
        reference_path
    ) as reference:

        height = reference.height
        width = reference.width

        transform = reference.transform
        crs = reference.crs

    with rasterio.open(
        source_path
    ) as source:

        aligned = np.zeros(
            (
                source.count,
                height,
                width
            ),
            dtype=np.float32
        )

        for band_index in range(
            source.count
        ):

            reproject(
                source=source.read(
                    band_index + 1
                ),
                destination=aligned[
                    band_index
                ],
                src_transform=source.transform,
                src_crs=source.crs,
                dst_transform=transform,
                dst_crs=crs,
                resampling=resampling_method
            )

    return aligned


# ============================================================
# Clean invalid values
# ============================================================

def clean_data(data):
    """
    Replace NaN and infinite values with zero.
    """

    return np.nan_to_num(
        data,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    ).astype(np.float32)


# ============================================================
# Normalize input channels
# ============================================================

def normalize_channels(data):
    """
    Standardize each input channel independently.

        normalized = (x - mean) / std

    This function is used for model INPUT features.

    The target LST is not normalized here because the target
    should remain in its physical temperature units.
    """

    normalized = np.zeros_like(
        data,
        dtype=np.float32
    )

    for channel_index in range(
        data.shape[0]
    ):

        channel = data[
            channel_index
        ]

        valid = np.isfinite(
            channel
        )

        if not np.any(valid):

            continue

        mean = np.mean(
            channel[valid]
        )

        std = np.std(
            channel[valid]
        )

        if std < 1e-8:

            std = 1.0

        normalized_channel = (
            channel - mean
        ) / std

        normalized_channel[
            ~valid
        ] = 0.0

        normalized[
            channel_index
        ] = normalized_channel

    return normalized


# ============================================================
# Crop arrays to common size
# ============================================================

def crop_to_common_size(*arrays):
    """
    Crop all arrays to the same height and width.

    All rasters should already be spatially aligned before
    this function is used.
    """

    height = min(
        array.shape[-2]
        for array in arrays
    )

    width = min(
        array.shape[-1]
        for array in arrays
    )

    cropped = []

    for array in arrays:

        cropped.append(
            array[
                ...,
                :height,
                :width
            ]
        )

    return cropped


# ============================================================
# Create patches
# ============================================================

def create_patches(
    data,
    patch_size=PATCH_SIZE,
    stride=STRIDE
):
    """
    Split a raster into fixed-size patches.

    Input:

        (channels, height, width)

    Output:

        (N, channels, patch_size, patch_size)
    """

    channels, height, width = data.shape

    patches = []

    for row in range(
        0,
        height - patch_size + 1,
        stride
    ):

        for column in range(
            0,
            width - patch_size + 1,
            stride
        ):

            patch = data[
                :,
                row:row + patch_size,
                column:column + patch_size
            ]

            patches.append(
                patch
            )

    if not patches:

        raise ValueError(
            "No patches were created. "
            f"The raster must be at least "
            f"{patch_size} x {patch_size} pixels."
        )

    return np.stack(
        patches
    ).astype(np.float32)


# ============================================================
# Prepare Sentinel-2
# ============================================================

def prepare_sentinel(
    sentinel_path
):
    """
    Load the eight Sentinel-2 feature channels:

        B2
        B3
        B4
        B8
        B11
        B12
        NDVI
        NDBI
    """

    sentinel, _ = read_raster(
        sentinel_path
    )

    if sentinel.shape[0] != 8:

        raise ValueError(
            "Expected 8 Sentinel-2 feature channels "
            "(B2, B3, B4, B8, B11, B12, NDVI, NDBI), "
            f"but found {sentinel.shape[0]}."
        )

    sentinel = clean_data(
        sentinel
    )

    sentinel = normalize_channels(
        sentinel
    )

    return sentinel


# ============================================================
# Prepare complete dataset
# ============================================================

def prepare_data(
    sentinel_path=None,
    ecostress_path=None,
    elevation_path=None,
    target_path=None,
    conductivity_path=None
):
    """
    Prepare the full dataset.

    Returns:

        X:
            (N, 10, 256, 256)

        Y:
            (N, 1, 256, 256)

        K:
            (N, 1, 256, 256)
    """

    # --------------------------------------------------------
    # Locate files
    # --------------------------------------------------------

    if sentinel_path is None:

        sentinel_path = select_file(
            SENTINEL_DIR,
            "Sentinel-2"
        )

    if ecostress_path is None:

        ecostress_path = select_file(
            ECOSTRESS_DIR,
            "ECOSTRESS"
        )

    if elevation_path is None:

        elevation_path = select_file(
            ELEVATION_DIR,
            "Elevation"
        )

    if target_path is None:

        target_path = select_file(
            TARGET_DIR,
            "Target LST"
        )

    if conductivity_path is None:

        conductivity_path = select_file(
            CONDUCTIVITY_DIR,
            "Conductivity"
        )


    # --------------------------------------------------------
    # Sentinel-2
    # --------------------------------------------------------

    print(
        "\nLoading Sentinel-2..."
    )

    sentinel = prepare_sentinel(
        sentinel_path
    )

    print(
        "Sentinel shape:",
        sentinel.shape
    )


    # --------------------------------------------------------
    # NASADEM
    # --------------------------------------------------------

    print(
        "\nAligning NASADEM..."
    )

    elevation = align_to_reference(
        elevation_path,
        sentinel_path,
        Resampling.bilinear
    )

    elevation = clean_data(
        elevation
    )

    elevation = normalize_channels(
        elevation
    )

    if elevation.shape[0] != 1:

        elevation = elevation[:1]

    print(
        "Elevation shape:",
        elevation.shape
    )


    # --------------------------------------------------------
    # ECOSTRESS
    # --------------------------------------------------------

    print(
        "\nAligning ECOSTRESS..."
    )

    ecostress = align_to_reference(
        ecostress_path,
        sentinel_path,
        Resampling.bilinear
    )

    ecostress = clean_data(
        ecostress
    )

    ecostress = normalize_channels(
        ecostress
    )

    if ecostress.shape[0] != 1:

        ecostress = ecostress[:1]

    print(
        "ECOSTRESS shape:",
        ecostress.shape
    )


    # --------------------------------------------------------
    # Target LST
    # --------------------------------------------------------

    print(
        "\nAligning target LST..."
    )

    target = align_to_reference(
        target_path,
        sentinel_path,
        Resampling.bilinear
    )

    target = clean_data(
        target
    )

    if target.shape[0] != 1:

        raise ValueError(
            "Target LST must contain exactly one band."
        )

    print(
        "Target LST shape:",
        target.shape
    )


    # --------------------------------------------------------
    # Conductivity
    # --------------------------------------------------------

    print(
        "\nAligning conductivity..."
    )

    conductivity = align_to_reference(
        conductivity_path,
        sentinel_path,
        Resampling.nearest
    )

    conductivity = clean_data(
        conductivity
    )

    if conductivity.shape[0] != 1:

        raise ValueError(
            "Conductivity must contain exactly one band."
        )

    print(
        "Conductivity shape:",
        conductivity.shape
    )


    # --------------------------------------------------------
    # Match spatial dimensions
    # --------------------------------------------------------

    (
        sentinel,
        elevation,
        ecostress,
        target,
        conductivity
    ) = crop_to_common_size(
        sentinel,
        elevation,
        ecostress,
        target,
        conductivity
    )


    # --------------------------------------------------------
    # Combine the 10 U-Net input channels
    #
    # 8 Sentinel-2
    # + 1 NASADEM
    # + 1 ECOSTRESS
    # --------------------------------------------------------

    X = np.concatenate(
        [
            sentinel,
            elevation,
            ecostress
        ],
        axis=0
    )


    # --------------------------------------------------------
    # Verify channel count
    # --------------------------------------------------------

    if X.shape[0] != 10:

        raise ValueError(
            "U-Net expects exactly 10 input channels, "
            f"but preprocessing produced {X.shape[0]}."
        )


    # --------------------------------------------------------
    # Create identical patches
    # --------------------------------------------------------

    print(
        "\nCreating patches..."
    )

    X = create_patches(
        X
    )

    Y = create_patches(
        target
    )

    K = create_patches(
        conductivity
    )


    # --------------------------------------------------------
    # Final consistency checks
    # --------------------------------------------------------

    if X.shape[0] != Y.shape[0]:

        raise ValueError(
            "Input and target patch counts do not match."
        )

    if X.shape[0] != K.shape[0]:

        raise ValueError(
            "Input and conductivity patch counts do not match."
        )


    expected_x_shape = (
        10,
        PATCH_SIZE,
        PATCH_SIZE
    )

    expected_y_shape = (
        1,
        PATCH_SIZE,
        PATCH_SIZE
    )

    if X.shape[1:] != expected_x_shape:

        raise ValueError(
            f"Unexpected X shape: {X.shape}"
        )

    if Y.shape[1:] != expected_y_shape:

        raise ValueError(
            f"Unexpected Y shape: {Y.shape}"
        )

    if K.shape[1:] != expected_y_shape:

        raise ValueError(
            f"Unexpected K shape: {K.shape}"
        )


    # --------------------------------------------------------
    # Print final shapes
    # --------------------------------------------------------

    print(
        "\nFinal dataset:"
    )

    print(
        "X:",
        X.shape
    )

    print(
        "Y:",
        Y.shape
    )

    print(
        "K:",
        K.shape
    )


    return X, Y, K


# ============================================================
# Save processed dataset
# ============================================================

def save_processed_data(
    X,
    Y,
    K
):
    """
    Save processed arrays to Google Drive.
    """

    PROCESSED_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    np.save(
        PROCESSED_DIR / "X.npy",
        X
    )

    np.save(
        PROCESSED_DIR / "Y.npy",
        Y
    )

    np.save(
        PROCESSED_DIR / "K.npy",
        K
    )

    print(
        "\nSaved processed data:"
    )

    print(
        PROCESSED_DIR / "X.npy"
    )

    print(
        PROCESSED_DIR / "Y.npy"
    )

    print(
        PROCESSED_DIR / "K.npy"
    )


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":

    print(
        "\nAvailable files:"
    )

    show_available_files()

    print(
        "\n"
        "The preprocessing module is ready. "
        "Call prepare_data() once the target-LST "
        "and conductivity GeoTIFFs are available."
    )